In [3]:
# Cell 1: PyTorch memory allocator setting, GPU check and Google Drive mount
# The allocator setting reduces GPU memory fragmentation; it must be set before PyTorch uses the GPU
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from google.colab import drive

print("GPU available:", torch.cuda.is_available())
print("Name:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

GPU available: True
Name: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cell 1b: GPU memory in use after the restart (expected: a few hundred MiB at most)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB


In [3]:
# Cell 2: Python version and TerraTorch installation
import sys
print("Python:", sys.version)

# TerraTorch provides the Prithvi backbones, the UperNet decoder and the model factory
!pip install -q terratorch rasterio

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.6/656.6 kB 15.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 97.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Cell 3: Import check and library versions (versions read from the installed package records)
from importlib.metadata import version
import torch
import terratorch

print("TerraTorch:", version("terratorch"))
print("PyTorch:", torch.__version__)
print("numpy:", version("numpy"))
print("CUDA available:", torch.cuda.is_available())

TerraTorch: 1.2.13
PyTorch: 2.11.0+cu130
numpy: 2.5.3
CUDA available: True


In [5]:
# Cell 4: Normalisation values in the official TerraTorch Sen1Floods11 data module
import inspect
from terratorch.datamodules import sen1floods11 as sen1floods11_dm

source_lines = inspect.getsource(sen1floods11_dm).splitlines()
for number, line in enumerate(source_lines, start=1):
    if any(word in line.lower() for word in ["mean", "std", "bands", "constant_scale"]):
        print(f"{number:4d}: {line}")

  16: MEANS = {
  32: STDS = {
  57:         bands: Sequence[str] = Sen1Floods11NonGeo.all_band_names,
  63:         constant_scale: float = 0.0001,
  76:             bands (Sequence[str], optional): List of bands to use. Defaults to Sen1Floods11NonGeo.all_band_names.
  82:             constant_scale (float, optional): Scale constant applied to the dataset. Defaults to 0.0001.
  91:         means = [MEANS[b] for b in bands]
  92:         stds = [STDS[b] for b in bands]
  93:         self.bands = bands
  98:         self.aug = AugmentationSequential(K.Normalize(means, stds), data_keys=None)
 100:         self.constant_scale = constant_scale
 116:                 bands=self.bands,
 117:                 constant_scale=self.constant_scale,
 127:                 bands=self.bands,
 128:                 constant_scale=self.constant_scale,
 138:                 bands=self.bands,
 139:                 constant_scale=self.constant_scale,
 149:                 bands=self.bands,
 150:             

In [4]:
# Cell 5: Prithvi-EO-2.0-300M-TL with the TerraTorch 1.x Sen1Floods11 example settings; GPU memory and speed test
import time
import torch.nn as nn
from terratorch.models import EncoderDecoderFactory

device = torch.device("cuda")
BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]
BATCH = 8

def build_prithvi(freeze_backbone):
    """Prithvi-EO-2.0-300M-TL backbone (pretrained, 512 x 512 input) with a UNet decoder, as in the example config."""
    model = EncoderDecoderFactory().build_model(
        task="segmentation",
        backbone="prithvi_eo_v2_300_tl",
        backbone_pretrained=True,
        backbone_img_size=512,
        backbone_coords_encoding=[],
        backbone_bands=BANDS,
        necks=[{"name": "SelectIndices", "indices": [5, 11, 17, 23]},
               {"name": "ReshapeTokensToImage"},
               {"name": "LearnedInterpolateToPyramidal"}],
        decoder="UNetDecoder",
        decoder_channels=[512, 256, 128, 64],
        head_dropout=0.1,
        num_classes=2,
    )
    if freeze_backbone:
        for p in model.encoder.parameters():
            p.requires_grad = False
    return model

ce_loss = nn.CrossEntropyLoss(ignore_index=-1)

for name, freeze in [("frozen", True), ("full", False)]:
    model, optimizer = None, None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        model = build_prithvi(freeze).to(device)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.1)
        scaler = torch.amp.GradScaler("cuda")

        # Three training steps on random input (batch 8, 6 bands, 512 x 512) and random labels
        x = torch.randn(BATCH, 6, 512, 512, device=device)
        y = torch.randint(0, 2, (BATCH, 512, 512), device=device)
        step_times = []
        for step in range(3):
            t0 = time.time()
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(x)
            logits = out.output if hasattr(out, "output") else out
            loss = ce_loss(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            step_times.append(time.time() - t0)

        print(f"{name}: parameters = {total / 1e6:.0f} M, trainable = {trainable / 1e6:.1f} M")
        print(f"    output shape = {tuple(logits.shape)}")
        print(f"    peak GPU memory = {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
        print(f"    step time (after warm-up) = {sum(step_times[1:]) / 2:.2f} s, "
              f"estimated epoch (31 steps) = {31 * sum(step_times[1:]) / 2:.0f} s")
    except torch.cuda.OutOfMemoryError:
        print(f"{name}: OUT OF GPU MEMORY at batch {BATCH} and 512 x 512")
    finally:
        del model, optimizer
        torch.cuda.empty_cache()

frozen: parameters = 324 M, trainable = 20.3 M
    output shape = (8, 2, 512, 512)
    peak GPU memory = 2.8 GB
    step time (after warm-up) = 0.55 s, estimated epoch (31 steps) = 17 s
full: parameters = 324 M, trainable = 324.2 M
    output shape = (8, 2, 512, 512)
    peak GPU memory = 14.5 GB
    step time (after warm-up) = 1.64 s, estimated epoch (31 steps) = 51 s


### Part C

In [5]:
# Cell 1 (new first lines): PyTorch memory allocator setting that reduces GPU memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
# Cell 6: Repository update and package import path
import sys
from pathlib import Path

REPO_URL = "https://github.com/Prhm93/floodfm-sen1floods11.git"
REPO_DIR = Path("/content/floodfm-sen1floods11")

# Clone on the first run in a session; pull the latest commits on later runs
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# The src folder is added to the Python search path, so that "import floodfm" finds the package
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

Already up to date.


In [7]:
# Cell 7: Dataset copy from Google Drive to the local Colab disk (skipped when the local copy is complete)
import shutil
import time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")
LOCAL_ROOT = Path("/content/data/sen1floods11")
FOLDERS = ["HandLabeled/S1Hand", "HandLabeled/S2Hand", "HandLabeled/LabelHand", "splits/flood_handlabeled"]

def local_copy_complete():
    """Returns True if each of the three layer folders on the local disk contains 446 .tif files."""
    return all(len(list((LOCAL_ROOT / f).glob("*.tif"))) == 446 for f in FOLDERS[:3])

if local_copy_complete():
    print("Local copy already complete.")
else:
    t0 = time.time()
    for folder in FOLDERS:
        shutil.copytree(DRIVE_ROOT / folder, LOCAL_ROOT / folder, dirs_exist_ok=True)
    print(f"Copy time: {time.time() - t0:.0f} s")

# Number of files in each copied folder
for folder in FOLDERS:
    print(f"{folder}: {len(list((LOCAL_ROOT / folder).iterdir()))} files")

Local copy already complete.
HandLabeled/S1Hand: 446 files
HandLabeled/S2Hand: 446 files
HandLabeled/LabelHand: 446 files
splits/flood_handlabeled: 4 files


In [8]:
# Cell 8: Short test run (frozen, seed 0, 2 epochs) in the separate test folder
import importlib
import floodfm.prithvi
importlib.reload(floodfm.prithvi)
from floodfm.prithvi import train_prithvi

TEST_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/checkpoints/_test")
history = train_prithvi(LOCAL_ROOT, TEST_ROOT, mode="frozen", seed=0, epochs=2)

epoch   1 | loss 0.3965 | valid loss 0.3058 | valid IoU 0.6315 | best 0.6315 | lr 1.0e-04 | 30.6 s
epoch   2 | loss 0.3418 | valid loss 0.2766 | valid IoU 0.6256 | best 0.6315 | lr 1.0e-04 | 33.8 s


In [9]:
# Cell 9: Three frozen Prithvi runs (seeds 0, 1, 2): training, then evaluation; safe to re-run after a disconnect
import subprocess
importlib.reload(floodfm.prithvi)
from floodfm.prithvi import train_prithvi, evaluate_prithvi

PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
CKPT_ROOT = PROJECT / "checkpoints"
RESULTS_DIR = PROJECT / "results"
PRED_DIR = PROJECT / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100
MODE = "frozen"

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

for seed in [0, 1, 2]:
    method = f"prithvi_{MODE}_seed{seed}"
    results_path = RESULTS_DIR / f"{method}.json"
    if results_path.exists():
        print(f"{method}: results file exists, skipped")
        continue

    print(f"\n===== {method}: training =====")
    train_prithvi(LOCAL_ROOT, CKPT_ROOT, mode=MODE, seed=seed, epochs=EPOCHS)

    print(f"===== {method}: evaluation =====")
    res = evaluate_prithvi(
        LOCAL_ROOT, CKPT_ROOT / method / "best.pt", MODE, method,
        description=(f"Prithvi-EO-2.0-300M-TL (TerraTorch), backbone {MODE}, UNet decoder, 6 Sentinel-2 bands, "
                     f"seed {seed}, {EPOCHS} epochs, best validation epoch; settings from the TerraTorch 1.x "
                     f"Sen1Floods11 example config; pixels without Sentinel-2 image predicted as not water"),
        results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
    )
    for split in ["test", "bolivia"]:
        s = res["splits"][split]["overall"]
        print(f"{method} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")


===== prithvi_frozen_seed0: training =====
epoch   1 | loss 0.3965 | valid loss 0.3058 | valid IoU 0.6319 | best 0.6319 | lr 1.0e-04 | 26.9 s
epoch   2 | loss 0.3418 | valid loss 0.2769 | valid IoU 0.6261 | best 0.6319 | lr 1.0e-04 | 32.8 s
epoch   3 | loss 0.3024 | valid loss 0.2645 | valid IoU 0.6257 | best 0.6319 | lr 1.0e-04 | 29.4 s
epoch   4 | loss 0.2735 | valid loss 0.2414 | valid IoU 0.6999 | best 0.6999 | lr 1.0e-04 | 31.1 s
epoch   5 | loss 0.2618 | valid loss 0.2349 | valid IoU 0.7006 | best 0.7006 | lr 1.0e-04 | 29.5 s
epoch   6 | loss 0.2475 | valid loss 0.2250 | valid IoU 0.6914 | best 0.7006 | lr 1.0e-04 | 31.5 s
epoch   7 | loss 0.2291 | valid loss 0.2121 | valid IoU 0.7572 | best 0.7572 | lr 1.0e-04 | 31.8 s
epoch   8 | loss 0.2119 | valid loss 0.1980 | valid IoU 0.7591 | best 0.7591 | lr 1.0e-04 | 33.3 s
epoch   9 | loss 0.2091 | valid loss 0.1810 | valid IoU 0.7428 | best 0.7591 | lr 1.0e-04 | 30.7 s
epoch  10 | loss 0.2051 | valid loss 0.1651 | valid IoU 0.7738 | 

: 

epoch  27 | loss 0.1152 | valid loss 0.1096 | valid IoU 0.8080 | best 0.8160 | lr 1.0e-04 | 32.3 s
epoch  28 | loss 0.1227 | valid loss 0.1105 | valid IoU 0.7998 | best 0.8160 | lr 1.0e-04 | 31.5 s
epoch  29 | loss 0.1106 | valid loss 0.1062 | valid IoU 0.8129 | best 0.8160 | lr 1.0e-04 | 31.5 s
epoch  30 | loss 0.1188 | valid loss 0.1042 | valid IoU 0.7958 | best 0.8160 | lr 1.0e-04 | 31.0 s
epoch  31 | loss 0.1127 | valid loss 0.1094 | valid IoU 0.8120 | best 0.8160 | lr 1.0e-04 | 32.6 s
epoch  32 | loss 0.1142 | valid loss 0.1022 | valid IoU 0.8135 | best 0.8160 | lr 1.0e-04 | 30.2 s
epoch  33 | loss 0.1055 | valid loss 0.0984 | valid IoU 0.8213 | best 0.8213 | lr 1.0e-04 | 31.7 s
epoch  34 | loss 0.1007 | valid loss 0.0952 | valid IoU 0.8183 | best 0.8213 | lr 1.0e-04 | 29.9 s
epoch  35 | loss 0.1036 | valid loss 0.0901 | valid IoU 0.8138 | best 0.8213 | lr 1.0e-04 | 31.9 s
epoch  36 | loss 0.1061 | valid loss 0.0929 | valid IoU 0.8147 | best 0.8213 | lr 1.0e-04 | 31.5 s
epoch  37 

In [6]:
# Cell 9b: GPU memory in use after the restart (expected: a few hundred MiB at most)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB


In [ ]:
# Cell 10: Prithvi full fine-tuning (seed 0): GPU memory check, training, then evaluation; safe to re-run after a disconnect
import importlib
import subprocess
from pathlib import Path
import floodfm.prithvi
importlib.reload(floodfm.prithvi)
from floodfm.prithvi import train_prithvi, evaluate_prithvi

# GPU memory in use before the run (must be near 0; other kernels on the same GPU reduce the free memory)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
CKPT_ROOT = PROJECT / "checkpoints"
RESULTS_DIR = PROJECT / "results"
PRED_DIR = PROJECT / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100
MODE = "full"

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

for seed in [0]:
    method = f"prithvi_{MODE}_seed{seed}"
    results_path = RESULTS_DIR / f"{method}.json"
    if results_path.exists():
        print(f"{method}: results file exists, skipped")
        continue

    print(f"\n===== {method}: training =====")
    # Full fine-tuning: micro-batches of 4 chips, weight update after 2 micro-batches (8 chips per update)
    train_prithvi(LOCAL_ROOT, CKPT_ROOT, mode=MODE, seed=seed, epochs=EPOCHS, accum_steps=2)

    print(f"===== {method}: evaluation =====")
    res = evaluate_prithvi(
        LOCAL_ROOT, CKPT_ROOT / method / "best.pt", MODE, method,
        description=(f"Prithvi-EO-2.0-300M-TL (TerraTorch), backbone {MODE}, UNet decoder, 6 Sentinel-2 bands, "
                     f"seed {seed}, batch 8 as 2 x 4 with gradient accumulation (T4 memory limit), "
                     f"{EPOCHS} epochs, best validation epoch; settings from the TerraTorch 1.x "
                     f"Sen1Floods11 example config; pixels without Sentinel-2 image predicted as not water"),
        results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
    )
    for split in ["test", "bolivia"]:
        s = res["splits"][split]["overall"]
        print(f"{method} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB

===== prithvi_full_seed0: training =====


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


epoch   1 | loss 0.4037 | valid loss 0.3103 | valid IoU 0.6618 | best 0.6618 | lr 1.0e-04 | 63.5 s
epoch   2 | loss 0.3558 | valid loss 0.2935 | valid IoU 0.5999 | best 0.6618 | lr 1.0e-04 | 66.0 s
epoch   3 | loss 0.3268 | valid loss 0.2726 | valid IoU 0.7251 | best 0.7251 | lr 1.0e-04 | 66.0 s
epoch   4 | loss 0.3033 | valid loss 0.2569 | valid IoU 0.6912 | best 0.7251 | lr 1.0e-04 | 68.2 s
epoch   5 | loss 0.2899 | valid loss 0.2543 | valid IoU 0.6640 | best 0.7251 | lr 1.0e-04 | 66.5 s
epoch   6 | loss 0.2716 | valid loss 0.2401 | valid IoU 0.7581 | best 0.7581 | lr 1.0e-04 | 66.5 s
epoch   7 | loss 0.2576 | valid loss 0.2104 | valid IoU 0.7740 | best 0.7740 | lr 1.0e-04 | 65.5 s
epoch   8 | loss 0.2401 | valid loss 0.2009 | valid IoU 0.7685 | best 0.7740 | lr 1.0e-04 | 65.7 s
epoch   9 | loss 0.2472 | valid loss 0.2034 | valid IoU 0.7568 | best 0.7740 | lr 1.0e-04 | 66.0 s
epoch  10 | loss 0.2388 | valid loss 0.2013 | valid IoU 0.7028 | best 0.7740 | lr 1.0e-04 | 68.0 s
epoch  11 

In [8]:
# Cell 11: Label-fraction test (10% of training chips, stratified by event): 6-band U-Net and frozen Prithvi, seeds 0, 1, 2
import importlib
import json
import subprocess
from pathlib import Path
import floodfm.data, floodfm.dataset, floodfm.train, floodfm.evaluate, floodfm.prithvi
for module in [floodfm.data, floodfm.dataset, floodfm.train, floodfm.evaluate, floodfm.prithvi]:
    importlib.reload(module)
from floodfm.data import label_fraction_subset
from floodfm.train import train
from floodfm.evaluate import evaluate_run
from floodfm.prithvi import train_prithvi, evaluate_prithvi

PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
CKPT_ROOT = PROJECT / "checkpoints"
RESULTS_DIR = PROJECT / "results"
PRED_DIR = PROJECT / "predictions"
STATS_PATH = RESULTS_DIR / "norm_stats.json"
PRED_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100
FRACTION = 0.10
TAG = "frac10"

# Fixed stratified subset: the same chips for both models and all seeds; saved to Drive as a record
subset = label_fraction_subset(LOCAL_ROOT, FRACTION, subset_seed=0)
subset_path = RESULTS_DIR / f"subset_{TAG}.json"
if not subset_path.exists():
    subset_path.write_text(json.dumps({"fraction": FRACTION, "subset_seed": 0, "chips": subset}, indent=2))
print(f"{TAG}: {len(subset)} training chips (of 252)")

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

def print_scores(method, res):
    """Prints the test and Bolivia IoU and F1 of one results dictionary."""
    for split in ["test", "bolivia"]:
        s = res["splits"][split]["overall"]
        print(f"{method} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")

for seed in [0, 1, 2]:
    # 6-band U-Net (random start) on the subset
    method = f"unet_s2_6_{TAG}_seed{seed}"
    results_path = RESULTS_DIR / f"{method}.json"
    if results_path.exists():
        print(f"{method}: results file exists, skipped")
    else:
        print(f"\n===== {method}: training =====")
        train(LOCAL_ROOT, STATS_PATH, CKPT_ROOT, modality="s2_6", seed=seed, epochs=EPOCHS,
              train_chip_ids=subset, run_name=method)
        print(f"===== {method}: evaluation =====")
        res = evaluate_run(
            LOCAL_ROOT, STATS_PATH, CKPT_ROOT / method / "best.pt", "s2_6", method,
            description=(f"U-Net (ResNet-34, no pretrained weights) on the six Prithvi bands, trained on "
                         f"{len(subset)} of 252 training chips ({TAG}, stratified by event, subset seed 0), "
                         f"252 samples per epoch, seed {seed}, {EPOCHS} epochs, best validation epoch"),
            results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
        )
        print_scores(method, res)

    # Frozen Prithvi on the same subset
    method = f"prithvi_frozen_{TAG}_seed{seed}"
    results_path = RESULTS_DIR / f"{method}.json"
    if results_path.exists():
        print(f"{method}: results file exists, skipped")
    else:
        print(f"\n===== {method}: training =====")
        train_prithvi(LOCAL_ROOT, CKPT_ROOT, mode="frozen", seed=seed, epochs=EPOCHS,
                      train_chip_ids=subset, run_name=method)
        print(f"===== {method}: evaluation =====")
        res = evaluate_prithvi(
            LOCAL_ROOT, CKPT_ROOT / method / "best.pt", "frozen", method,
            description=(f"Prithvi-EO-2.0-300M-TL (TerraTorch), backbone frozen, UNet decoder, 6 Sentinel-2 bands, "
                         f"trained on {len(subset)} of 252 training chips ({TAG}, stratified by event, subset seed 0), "
                         f"252 samples per epoch, seed {seed}, {EPOCHS} epochs, best validation epoch"),
            results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
        )
        print_scores(method, res)

frac10: 26 training chips (of 252)

===== unet_s2_6_frac10_seed0: training =====
epoch   1 | loss 0.9806 | valid IoU 0.5598 | best 0.5598 | 19.1 s
epoch   2 | loss 0.4522 | valid IoU 0.7789 | best 0.7789 | 16.9 s
epoch   3 | loss 0.3279 | valid IoU 0.7605 | best 0.7789 | 18.1 s
epoch   4 | loss 0.2769 | valid IoU 0.7274 | best 0.7789 | 17.7 s
epoch   5 | loss 0.2910 | valid IoU 0.7726 | best 0.7789 | 17.4 s
epoch   6 | loss 0.2578 | valid IoU 0.7889 | best 0.7889 | 17.5 s
epoch   7 | loss 0.2350 | valid IoU 0.7597 | best 0.7889 | 17.5 s
epoch   8 | loss 0.2072 | valid IoU 0.7798 | best 0.7889 | 18.2 s
epoch   9 | loss 0.2027 | valid IoU 0.7283 | best 0.7889 | 17.1 s
epoch  10 | loss 0.2062 | valid IoU 0.7252 | best 0.7889 | 16.7 s
epoch  11 | loss 0.2068 | valid IoU 0.7923 | best 0.7923 | 16.5 s
epoch  12 | loss 0.1856 | valid IoU 0.7923 | best 0.7923 | 18.0 s
epoch  13 | loss 0.1823 | valid IoU 0.7633 | best 0.7923 | 18.3 s
epoch  14 | loss 0.2028 | valid IoU 0.8074 | best 0.8074 | 19

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Prithvi_EO_V2_300M_TL.pt: reconstructing file:   0%|          |  0.00B / 1.33GB            

Prithvi_EO_V2_300M_TL.pt: downloading bytes:           |  0.00B            

epoch   1 | loss 0.4274 | valid loss 0.3326 | valid IoU 0.5556 | best 0.5556 | lr 1.0e-04 | 31.1 s
epoch   2 | loss 0.3433 | valid loss 0.2960 | valid IoU 0.6501 | best 0.6501 | lr 1.0e-04 | 31.6 s
epoch   3 | loss 0.3204 | valid loss 0.2760 | valid IoU 0.6545 | best 0.6545 | lr 1.0e-04 | 31.6 s
epoch   4 | loss 0.2999 | valid loss 0.2624 | valid IoU 0.6599 | best 0.6599 | lr 1.0e-04 | 31.9 s
epoch   5 | loss 0.2813 | valid loss 0.2379 | valid IoU 0.7165 | best 0.7165 | lr 1.0e-04 | 31.1 s
epoch   6 | loss 0.2603 | valid loss 0.2238 | valid IoU 0.7338 | best 0.7338 | lr 1.0e-04 | 31.7 s
epoch   7 | loss 0.2430 | valid loss 0.2088 | valid IoU 0.7217 | best 0.7338 | lr 1.0e-04 | 29.2 s
epoch   8 | loss 0.2213 | valid loss 0.1978 | valid IoU 0.7279 | best 0.7338 | lr 1.0e-04 | 31.0 s
epoch   9 | loss 0.2078 | valid loss 0.1991 | valid IoU 0.7387 | best 0.7387 | lr 1.0e-04 | 31.1 s
epoch  10 | loss 0.1929 | valid loss 0.1873 | valid IoU 0.7250 | best 0.7387 | lr 1.0e-04 | 31.8 s
epoch  11 